In [1]:
# notebook config

# path to the pytorch pth weights file downloaded from Meta
LLAMA31_PT_MODEL_PATH = "/Users/cuongwilliams/.llama/checkpoints/Meta-Llama3.1-8B-Instruct/consolidated.00.pth"

# path to the tokenizer model file downloaded from Meta
LLAMA31_PT_TOKENIZER_PATH = "/Users/cuongwilliams/.llama/checkpoints/Meta-Llama3.1-8B/tokenizer.model"

# path to Hugging Face tokenizer files
#LLAMA31_HF_TOKENIZER_PATH = "/Users/cuongwilliams/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3.1-8B/snapshots/48d6d0fc4e02fb1269b36940650a1b7233035cbb/tokenizer.json"
LLAMA31_HF_TOKENIZER_PATH = "/Users/cuongwilliams/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3.1-8B-Instruct/snapshots/5206a32e0bd3067aef1ce90f5528ade7d866253f/tokenizer.json"
LLAMA31_HF_TOKENIZER_CONFIG_PATH = "/Users/cuongwilliams/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3.1-8B-Instruct/snapshots/5206a32e0bd3067aef1ce90f5528ade7d866253f/tokenizer_config.json"

# path to my bytesarray exports
LLAMA31_HF_MODEL_TOC = "/tmp/llama_toc.json"
LLAMA31_HF_MODEL_BIN = "/tmp/llama.bin"

In [30]:
# imports

# built-in
import pickle
import zipfile
import io
import json
import base64
import numpy as np
import importlib
from collections import defaultdict
from collections import UserDict, OrderedDict
from typing import TYPE_CHECKING, Any, Dict, List, NamedTuple, Optional, Sequence, Tuple, Union
import math
import types

# external
from tokenizers import Tokenizer, AddedToken, Encoding
import ml_dtypes


In [38]:
# dynamic imports

# create torch "stubs" dynamically (needed to unpickle python tensors in pth model file )
class BFloat16Storage:
    dtype = ml_dtypes.bfloat16
    nbytes = 2
def _rebuild_tensor_v2(a,b,c,d,e,f):
    '''Stub for the pickled pytorch call.'''
    #print("rebuild!",a,b,c,d,e,f)
    tensor = a.reshape(c)
    return tensor
torch = types.ModuleType("torch")
#print(torch, dir(torch))
torch.BFloat16Storage = BFloat16Storage
print(torch.BFloat16Storage)
torch._rebuild_tensor_v2 = _rebuild_tensor_v2
#print(torch._rebuild_tensor_v2)

<class '__main__.BFloat16Storage'>


In [39]:
HF_TORCH_KEY_MAPPING = None
TORCH_HF_KEY_MAPPING = None
NUM_LAYERS = None

In [40]:

def create_tokenizer():
    global TOKENIZER
    # create tokenizer from configs 
    f = open(LLAMA31_HF_TOKENIZER_CONFIG_PATH,"r")
    contents = f.read()
    f.close()
    tokenizer_config = json.loads(contents)
    items = tokenizer_config["added_tokens_decoder"].items()
    tokens = []
    for idx, token in items:
        if isinstance(token, dict):
            token = AddedToken(**token)
            tokens.append(token)
    tokenizer = Tokenizer.from_file(LLAMA31_HF_TOKENIZER_PATH)
    tokenizer.no_truncation()
    tokenizer.add_tokens( tokens )
    return tokenizer
    

In [41]:

def create_model_key_mappings():
    
    global HF_TORCH_KEY_MAPPING, TORCH_HF_KEY_MAPPING, NUM_LAYERS
    
    # HF <-> TORCH
    HF_TORCH_KEY_MAPPING = {
        'model.embed_tokens.weight': 'tok_embeddings.weight', 
        'model.layers.*.input_layernorm.weight': 'layers.*.attention_norm.weight', 
        'model.layers.*.mlp.down_proj.weight': 'layers.*.feed_forward.w2.weight', 
        'model.layers.*.mlp.gate_proj.weight': 'layers.*.feed_forward.w1.weight', 
        'model.layers.*.mlp.up_proj.weight': 'layers.*.feed_forward.w3.weight', 
        'model.layers.*.post_attention_layernorm.weight': 'layers.*.ffn_norm.weight', 
        'model.layers.*.self_attn.k_proj.weight': 'layers.*.attention.wk.weight',
        'model.layers.*.self_attn.o_proj.weight': 'layers.*.attention.wo.weight', 
        'model.layers.*.self_attn.q_proj.weight': 'layers.*.attention.wq.weight' , 
        'model.layers.*.self_attn.v_proj.weight': 'layers.*.attention.wv.weight',
        'lm_head.weight': 'output.weight', 
        'model.norm.weight': 'norm.weight'
    }
    
    # TORCH <-> HF
    NUM_LAYERS=32
    TORCH_HF_KEY_MAPPING = OrderedDict()
    for item in HF_TORCH_KEY_MAPPING.items():
        hf_key_pre = item[0]
        torch_key_pre = item[1]
        wild = hf_key_pre.find("*")
        if wild >0:      
            for i in range(NUM_LAYERS):
                hf_key = hf_key_pre[:wild] + str(i) + hf_key_pre[wild+1:]
                loc = item[1].find( "*" )
                torch_key = item[1][:loc] + str(i) + item[1][loc+1:]
                TORCH_HF_KEY_MAPPING[torch_key] = hf_key
        else:
            TORCH_HF_KEY_MAPPING[torch_key_pre] = hf_key_pre


In [42]:
# tokenize a test prompt

# TODO: 
#   * minify the code further
#   * figure out why extra start token

def tokenize_prompt( prompt, tokenizer ):
    encoding = tokenizer.encode_batch(prompt)[0]
    encodings = [ encoding ]
    
    def convert_encoding(encoding):
        encodings = [ encoding ]
        encoding_dict = defaultdict(list)
        return_token_type_ids= False 
        return_attention_mask= True 
        return_special_tokens_mask= False 
        return_offsets_mapping= False 
        return_length= False
        for e in encodings:
            encoding_dict["input_ids"].append(e.ids)
        
            if return_token_type_ids:
                encoding_dict["token_type_ids"].append(e.type_ids)
            if return_attention_mask:
                encoding_dict["attention_mask"].append(e.attention_mask)
            if return_special_tokens_mask:
                encoding_dict["special_tokens_mask"].append(e.special_tokens_mask)
            if return_offsets_mapping:
                encoding_dict["offset_mapping"].append(e.offsets)
            if return_length:
                encoding_dict["length"].append(len(e.ids))
        return encoding_dict, encodings
      
    tokens_and_encodings = [ convert_encoding(encoding) for encoding in encodings ]  

    sanitized_tokens = {}
    for key in tokens_and_encodings[0][0].keys():
        stack = [e for item, _ in tokens_and_encodings for e in item[key]]
        sanitized_tokens[key] = stack
    sanitized_encodings = [e for _, item in tokens_and_encodings for e in item]
    
    # HACK: figure out why start token repeats
    token_ids = np.array( sanitized_tokens['input_ids'], np.uint32 )[:,1:]  
    attention_mask = np.ma.make_mask( sanitized_tokens['attention_mask'])[:,1:]
    
    #print("token_ids=",token_ids)
    #print("attention_mask=", attention_mask)
    #print("sizes=", token_ids.size, attention_mask.size)

    return token_ids, 1, token_ids.shape[1]

In [43]:
# load the hugging face tensor exported model

def load_hf_export_model():
    #print("reading llama_toc")
    with open(LLAMA31_HF_MODEL_TOC,"r") as f:
        llama_toc = json.loads(f.read())
    #print("llama_toc=", llama_toc)  
    #print("reading llama_bin")
    with open(LLAMA31_HF_MODEL_BIN,"rb") as f:
        llama_b = f.read()
    #print("llama_bin size=", len(llama_b))  
    llama_io = io.BytesIO(llama_b)
    fcounter=0
    hf_model = OrderedDict()
    for item in llama_toc.items():
        
        # Get the HF exported model key name
        hf_key = item[0]  
        shape, tot_size = item[1] # get the tensor shape and total size captured during export
        array_buf = llama_io.read(tot_size) # read just the total bytes for the tensor
    
        # Convert the buffer ato  file stream object
        iobuff = io.BytesIO(array_buf)
        iobuff.seek(0)

        # Prepare class for reading the torch save format (zip archive)
        class CustomLoader:
            def __init__(self,f):
                self.f = f # file pointer
                self.zf = None # pointer to zipfile object
                self.unpickler = None # custom unpickler
            def load(self):
                # process file as zipped compressed
                izf = zipfile.is_zipfile(self.f)
                if not izf: raise Exception("ERROR: This loader only supports recent Pytorch versions")
                self.zf = zipfile.ZipFile(self.f)
                #print("zip names=", self.zf.namelist())
                # unzip the TOC section
                toc_bytes = io.BytesIO( self.zf.read("archive/data.pkl") )                 
                def persistent_load(saved_id):
                    typename = saved_id[0]           
                    if typename == 'storage':
                        storage_type, key, location, numel = saved_id[1:]
                        name = f"archive/data/{key}"
                    else:
                        raise Exception("unknown storage")              
                    dp = io.BytesIO( self.zf.read(name) )
                    bytes = dp.read()
                    tot_bytes = numel * 2 # assume bfloat16         
                    #print("bytes=",len(bytes), key, numel, tot_bytes)
                    arr = np.frombuffer(bytes, 
                                        dtype=ml_dtypes.bfloat16, # assume bfloat16
                                        count=numel)
                    return arr     
                class UnpicklerWrapper(pickle.Unpickler):
                    def find_class(self, mod_name, name):
                        #print("GW _legacy_load unpickler find_class", 
                        #      mod_name, name,
                        #      type(mod_name), type(name))
                        return super().find_class(mod_name, name)
                    def __init__(self,f):
                        self.f = f
                        super(UnpicklerWrapper, self).__init__(f)
                # Unpickle the TOC object which creates the unpacked model
                self.unpickler = UnpicklerWrapper(toc_bytes)
                self.unpickler.persistent_load = persistent_load    
                result = self.unpickler.load()
                return result
        # read the tensor associated with the HF key
        loader = CustomLoader(iobuff)
        arr = loader.load()
        #print("arr=", arr.shape, arr.dtype)  
        fcounter += tot_size
        hf_model[hf_key] = arr # replace key's value with array
        
    return hf_model

#print("Done.")

In [44]:
# load the model from the pth file via custom loader class

def load_torch_model():
    class PthLoader:
        def __init__(self,f):
            self.f = f # file pointer
            self.zf = None # pointer to zipfile object
            self.unpickler = None # custom unpickler
            self.model = [] # holds the tensor objects
        def load(self):
            # process file as zipped compressed
            izf = zipfile.is_zipfile(self.f)
            if not izf: raise Exception("ERROR: This loader only supports recent Pytorch versions")
            self.zf = zipfile.ZipFile(self.f)
            # unzip the TOC section
            toc_bytes = io.BytesIO( self.zf.read("consolidated.00/data.pkl") )
            def load_tensor(dtype, numel, tot_bytes, key, location):
                # extract the zipped contents
                name = f"consolidated.00/data/{key}"
                dp = io.BytesIO( self.zf.read(name) )
                bytes = dp.read()
                if len(bytes) != tot_bytes:
                    raise Exception("ERROR: Tensor size mismatch, expected %d got %d" % tot_bytes, len(bytes))
                # reconstitute numpy array from the bytes
                arr = np.frombuffer(bytes, 
                                    dtype=dtype,
                                    #dtype=np.float32,
                                    count=numel)
                return arr
            def persistent_load(saved_id):
                assert isinstance(saved_id, tuple) # require 'tuple' type
                # Only support 'storage' typename currently
                typename = saved_id[0] 
                assert (
                    typename == "storage"
                ), f"Unknown typename for persistent_load, expected 'storage' but got '{typename}'"
                # unpack storage metadata
                storage_type, key, location, numel = saved_id[1:]
                # unpack buffer as typed numpy array
                dtype = storage_type.dtype
                tot_bytes = numel * storage_type.nbytes
                typed_storage = load_tensor( dtype, numel, tot_bytes, key, location )
                return typed_storage
            class UnpicklerWrapper(pickle.Unpickler):
                def __init__(self,f):
                    self.f = f
                    super(UnpicklerWrapper, self).__init__(f)
            # Unpickle the TOC object which creates the unpacked model
            self.unpickler = UnpicklerWrapper(toc_bytes)
            self.unpickler.persistent_load = persistent_load    
            result = self.unpickler.load()
            return result
    f = open( LLAMA31_PT_MODEL_PATH, "rb" )
    loader = PthLoader(f)
    torch_model = loader.load()
    return torch_model

#print("Done.")

In [45]:
# Sync the model with the HF export

def compare_models(stop_key):
    # check tensors between HF export and PTH model
    for item in torch_model.items():
        torch_key = item[0]
        #print("torch_key=", torch_key)
        hf_key = TORCH_HF_KEY_MAPPING[torch_key]
        #print("hf_key=", hf_key)
        #print("keys", "torch=", torch_key, "hf_key=", hf_key)
        torch_tensor = torch_model[torch_key]
        hf_tensor = hf_model[hf_key]
        #print("torch_tensor=", torch_tensor)
        #print("hf_tensor=", hf_tensor)
        if stop_key and hf_key == stop_key:
            print("found!")
            break


In [46]:
#
# token_ids X embedding_weights
#

def token_ids__2__token_embeddings( token_ids, hf_model ):
    # start with the prompt token ids
    #print("token_ids shape=", #token_ids, 
    #      token_ids.shape, token_ids.dtype)
    # get the embedding matrix
    embed_weights = hf_model[ TORCH_HF_KEY_MAPPING['tok_embeddings.weight'] ]
    #print("embed weights shape=", embed_weights.shape, embed_weights.dtype)
    # lookup embeddings using token_ids as index
    #print("idx=", token_ids.squeeze().shape )
    embedding_activations = embed_weights[ None, token_ids.squeeze(), :]
    #print("embedding activations=", #embedding_activations, 
    #      embedding_activations.shape, embedding_activations.dtype)
    return embedding_activations

#print("Done.")

In [47]:
#
# Get rotary embeddings
# 

def get_rotary(dtype):
    # default ROPE
    base= 500000.0 
    dim= 128
    inv_freq = 1.0 / (base ** (np.arange(0, dim, 2, dtype=np.int64) / dim))
    #print("default inv_freq=", inv_freq, inv_freq.shape, inv_freq.dtype)

    # llama3 ROPE parameters will yield new inv_freq
    factor=8.0
    low_freq_factor=1.0
    high_freq_factor=4.0
    old_context_len=8192
    low_freq_wavelen=8192.0
    high_freq_wavelen=2048.0
    wavelen = 2 * math.pi / inv_freq
    # wavelen < high_freq_wavelen: do nothing
    # wavelen > low_freq_wavelen: divide by factor
    inv_freq_llama = np.where(wavelen > low_freq_wavelen, inv_freq / factor, inv_freq)
    # otherwise: interpolate between the two, using a smooth factor
    smooth_factor = (old_context_len / wavelen - low_freq_factor) / (high_freq_factor - low_freq_factor)
    smoothed_inv_freq = (1 - smooth_factor) * inv_freq_llama / factor + smooth_factor * inv_freq_llama
    is_medium_freq = ~(wavelen < high_freq_wavelen) * ~(wavelen > low_freq_wavelen)
    inv_freq_llama = np.where(is_medium_freq, smoothed_inv_freq, inv_freq_llama)
    #print("llama3 inv_freq=", #inv_freq_llama, 
    #      inv_freq_llama.shape, inv_freq_llama.dtype)

    #
    # rotary forward() computation to get cos, sin matrices
    #
    position_ids_expanded = np.arange( token_ids.shape[1]  ).reshape(1, 1, token_ids.shape[1] ).astype(np.float32)
    #print("position_ids_expanded=", #position_ids_expanded, 
    #      position_ids_expanded.shape, position_ids_expanded.dtype )
    
    # TODO: not sure the following works with batch size dim > 1, HF transformer uses "expand"
    #HF inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1)
    inv_freq_expanded = inv_freq_llama.reshape(1, inv_freq_llama.shape[0], 1 ).astype(np.float32)
    #print("inv_freq_expanded=", #inv_freq_expanded, 
    #      inv_freq_expanded.shape, inv_freq_expanded.dtype)

    freqs = np.matmul( inv_freq_expanded, position_ids_expanded )
    #print("freqs=", #freqs, 
    #      freqs.shape, freqs.dtype)
    freqs_t = freqs.transpose((0,2,1))
    #print("freqs_t=", #freqs_t, 
    #      freqs_t.shape, freqs_t.dtype)
    emb = np.concatenate( [ freqs_t, freqs_t ], 2)
    #print("emb=", #emb, 
    #      emb.shape, emb.dtype)
    cos = np.cos( emb )
    #print("cos=", #cos, 
    #      cos.shape, cos.dtype)
    sin = np.sin( emb )
    #print("sin=", #sin, 
    #      sin.shape, sin.dtype)
    cos = cos.astype( dtype ) #embedding_activations.dtype)
    sin = sin.astype( dtype ) #embedding_activations.dtype)
    #print("cos=", #cos, 
    #      cos.shape, cos.dtype)
    #print("sin=", #sin, 
    #      sin.shape, sin.dtype)
    transformations = (cos, sin)

    return transformations
        
#print("Done.")

In [49]:
def apply_rms_norm_pre_attention( hidden_state, hf_model, layer ):
    residual = hidden_state
    hidden_state = hidden_state.astype(np.float32)
    #print("hidden_state_32=", #hidden_state, 
    #      hidden_state.dtype) 
    variance_epsilon= 1e-05
    rmsnorm = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.attention_norm.weight' % layer] ]
    #print("rmsnorm=", #rmsnorm, 
    #      rmsnorm.shape)
    variance = np.mean( np.power(hidden_state,2), 2, keepdims=True )
    #print("variance=", #variance, 
    #      variance.shape)
    v_sqrt = np.sqrt( variance + variance_epsilon )
    #print("v_sqrt=", #v_sqrt, 
    #      v_sqrt.shape)
    v_rsqrt = np.reciprocal( v_sqrt  )
    #print("v_rsqrt=", #v_rsqrt, 
    #      v_rsqrt.shape)
    hidden_state = np.multiply( hidden_state, v_rsqrt )
    #print("hidden_state_mult=", #hidden_state, 
    #      hidden_state.shape)
    normalized_hidden_state = rmsnorm * hidden_state.astype( ml_dtypes.bfloat16 )
    #print("apply_rms_norm_pre_attention: normalized_hidden_state=", #normalized_hidden_state, 
    #      normalized_hidden_state.shape,normalized_hidden_state.dtype)
    return residual, normalized_hidden_state

In [50]:
def apply_q_projection( normalized_hidden_state, hf_model, batch_size, q_len, num_heads, head_dim, layer ):
    hidden_state = normalized_hidden_state.squeeze()
    #print("hidden_state=", #hidden_state, 
    #      hidden_state.shape,hidden_state.dtype)
    q_weights = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.attention.wq.weight' % layer] ] 
    #print("q_weights=", #q_weights, 
    #      q_weights.shape, q_weights.dtype)  
    #q_proj = np.matmul( hidden_state, q_weights )
    q_proj = np.matmul( hidden_state, q_weights.transpose() )   # WE START TO DEVIATE SLIGHTLY HERE!!!!
    q_proj = q_proj.astype( hidden_state.dtype) # TODO: Why does matmul upconvert to fp32?
    #print("apply_q_projection:", hidden_state.dtype, q_weights.dtype, q_weights.transpose().dtype, q_proj.dtype)
    #print("q_proj=", #q_proj, 
    #      q_proj.shape, q_proj.dtype)
    q_proj_heads = q_proj.reshape(batch_size, q_len, num_heads, head_dim ).transpose((0,2,1,3))
    #print("apply_q_projection:", q_proj.dtype, q_proj_heads.dtype)
    #print("q_proj_heads=", #q_proj_heads, 
    #      q_proj_heads.shape, q_proj_heads.dtype)
    return q_proj_heads

In [51]:
def apply_k_projection( normalized_hidden_state, hf_model, batch_size, q_len, num_key_value_heads, head_dim, layer ):
    global TORCH_HF_KEY_MAPPING
    hidden_state = normalized_hidden_state.squeeze()
    #print("hidden_state=", #hidden_state, 
    #      hidden_state.shape,hidden_state.dtype)
    k_weights = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.attention.wk.weight' % layer ] ]
    #print("k_weights=", #q_weights, 
    #      k_weights.shape, k_weights.dtype)   
    k_proj = np.matmul( hidden_state, k_weights.transpose() )   # STARTS TO DEVIATE SLIGHTLY
    k_proj = k_proj.astype( hidden_state.dtype) # TODO: Why does matmul upconvert to fp32?
    #print("k_proj=", #k_proj, 
    #      k_proj.shape, k_proj.dtype)   
    k_proj_heads = k_proj.reshape(batch_size, q_len, num_key_value_heads, head_dim ).transpose((0,2,1,3))
    #print("k_proj_heads=", #k_proj_heads, 
    #      k_proj_heads.shape, k_proj_heads.dtype)
    return k_proj_heads

In [52]:
def apply_rotary( q_proj_heads, k_proj_heads, hf_model ):           
    #print("q_proj_heads=", #q_proj_heads, 
    #       q_proj_heads.shape, q_proj_heads.dtype)
    #print("k_proj_heads=", #k_proj_heads, 
    #       k_proj_heads.shape, k_proj_heads.dtype)
    dtype = q_proj_heads.dtype
    cos, sin = get_rotary( dtype ) 
    cos_sq = cos.reshape((1,cos.shape[0],cos.shape[1],cos.shape[2]))
    #print("cos_sq=", #cos_sq, 
    #      cos_sq.shape, cos_sq.dtype)
    sin_sq = sin.reshape((1,sin.shape[0],sin.shape[1],sin.shape[2]))
    #print("sin_sq=",#sin_sq, 
    #      sin_sq.shape, sin_sq.dtype)
    #print("pre q_embed", q_proj_heads.shape, cos_sq.shape, cos_sq.shape)
    q_embed_a = np.multiply( q_proj_heads, cos_sq ) 
    #print("q_embed_a=", #q_embed_a, 
    #      q_embed_a.shape)
    #    x1 = x[..., : x.shape[-1] // 2]
    #    x2 = x[..., x.shape[-1] // 2 :]
    #    return torch.cat((-x2, x1), dim=-1)
    x1 = q_proj_heads[:,:,:, :q_proj_heads.shape[-1] // 2 ]
    #print("q x1=", #x1,
    #      x1.shape, x1.dtype)
    x2 = q_proj_heads[:,:,:, q_proj_heads.shape[-1] // 2: ]
    #print("q x2=", #x2,
    #      x2.shape, x2.dtype)
    rotary_half = np.concatenate( [ np.negative(x2), x1 ], 3 )
    #print("rotary_half=", #rotary_half,
    #      rotary_half.shape, rotary_half.dtype)
    q_embed_b = np.multiply(rotary_half, sin_sq )
    #print("q_embed_b=", #q_embed_b,
    #      q_embed_b.shape)
    q_embed = q_embed_a + q_embed_b 
    #print("apply_rotary: q_embed=", #q_embed, 
    #      q_embed.shape)
    #print("pre k_embed", k_proj_heads.shape, cos_sq.shape, cos_sq.shape)
    k_embed_a = np.multiply( k_proj_heads, cos_sq ) 
    #print("k_embed_a=", #k_embed_a, 
    #      k_embed_a.shape)
    #    x1 = x[..., : x.shape[-1] // 2]
    #    x2 = x[..., x.shape[-1] // 2 :]
    #    return torch.cat((-x2, x1), dim=-1)
    x1 = k_proj_heads[:,:,:, :k_proj_heads.shape[-1] // 2 ]
    x2 = k_proj_heads[:,:,:, k_proj_heads.shape[-1] // 2: ]
    rotary_half = np.concatenate( [ np.negative(x2), x1 ], 3 )
    k_embed_b = np.multiply(rotary_half, sin_sq )
    #print("k_embed_b=", #k_embed_b
    #      k_embed_b.shape) 
    k_embed = k_embed_a + k_embed_b 
    #print("apply_rotary: k_embed=", #k_embed, 
    #      k_embed.shape)
    return q_embed, k_embed
#print("Done.")

In [53]:
def apply_v_projection( normalized_hidden_state, hf_model, batch_size, q_len, num_key_value_heads, head_dim, layer ):
    hidden_state = normalized_hidden_state.squeeze()
    #print("hidden_state =", #hidden_state, 
    #      hidden_state.shape,hidden_state.dtype)
    # value projection
    v_weights = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.attention.wv.weight' % layer] ]
    #print("v_weights=", #v_weights, 
    #      v_weights.shape, v_weights.dtype)
    v_proj = np.matmul( hidden_state, v_weights.transpose() )   # STARTS TO DEVIATE SLIGHTLY
    v_proj = v_proj.astype( hidden_state.dtype) # TODO: Why does matmul upconvert to fp32?   
    #print("v_proj=", #v_proj, 
    #      v_proj.shape, v_proj.dtype) 
    v_proj_heads = v_proj.reshape(batch_size, q_len, num_key_value_heads, head_dim ).transpose((0,2,1,3))
    #print("v_proj_heads=", #v_proj_heads, 
    #      v_proj_heads.shape, v_proj_heads.dtype)
    return v_proj_heads

In [54]:
# repeat kv

def repeat_kv( k_embed, v_proj, num_key_value_groups ):
    # k
    batch, num_key_value_heads, slen, head_dim = k_embed.shape
    k_embed_expanded = np.expand_dims(k_embed,2)
    #print("vk_embed_expanded=", #k_embed_heads_expanded, 
    #      k_embed_expanded.shape, k_embed_expanded.dtype)
    k_embed_repeat = np.broadcast_to( k_embed_expanded, 
            (batch, num_key_value_heads, num_key_value_groups, slen, head_dim) )
    #print("k_embed_repeat=", #k_embed_repeat, 
    #      k_embed_repeat.shape, k_embed_repeat.dtype)                                
    k_repeated = k_embed_repeat.reshape( (batch, num_key_value_heads * num_key_value_groups, slen, head_dim) )
    #print("k_states=", #k_states, 
    #      k_states.shape, k_states.dtype)  
    # v
    batch, num_key_value_heads, slen, head_dim = v_proj.shape
    v_proj_expanded = np.expand_dims(v_proj,2)
    #print("v_proj_heads_expanded=", #v_proj_heads_expanded, 
    #      v_proj_heads_expanded.shape, v_proj_heads_expanded.dtype)
    v_proj_repeat = np.broadcast_to( v_proj_expanded, 
            (batch, num_key_value_heads, num_key_value_groups, slen, head_dim) )
    #print("v_proj_heads_repeat=", #v_proj_heads_repeat, 
    #      v_proj_heads_repeat.shape, v_proj_heads_repeat.dtype)                                
    v_repeated = v_proj_repeat.reshape( (batch, num_key_value_heads * num_key_value_groups, slen, head_dim) )
    #print("v_states=", #v_states, 
    #      v_states.shape, v_states.dtype)  
    return k_repeated, v_repeated

In [55]:
def project_to_qkv( normalized_hidden_state, hf_model, batch_size, q_len, layer ):
    #batch_size=1 # TODO
    #q_len=token_ids.shape[1]
    num_heads=32
    head_dim=128
    num_key_value_heads=8
    num_key_value_groups = num_heads // num_key_value_heads
    q_proj_heads = apply_q_projection( normalized_hidden_state, hf_model, 
                            batch_size, q_len, num_heads, head_dim, layer )
    k_proj_heads = apply_k_projection( normalized_hidden_state, hf_model, 
                            batch_size, q_len, num_key_value_heads, head_dim, layer )
    q_rotated, k_rotated = apply_rotary( q_proj_heads, k_proj_heads, hf_model  )                                 
    v_proj = apply_v_projection( normalized_hidden_state, hf_model, 
                            batch_size, q_len, num_key_value_heads, head_dim, layer )
    #print("project_to_qkv", q_proj_heads.dtype, k_proj_heads.dtype, v_proj.dtype)
    k_repeated, v_repeated = repeat_kv( k_rotated, v_proj, num_key_value_groups )
    return q_rotated, k_repeated, v_repeated

In [56]:
# Apply attention

def apply_attention( q_states, k_states, v_states, hf_model, batch_size, q_len, layer ):
    #print("apply_attention:", q_states.dtype, k_states.dtype, v_states.dtype)
    L = q_states.shape[-2]
    S = k_states.shape[-2]
    #print("apply_attention: L,S", L, S)   
    scale_factor = 1 / math.sqrt(q_states.shape[-1])
    #print("scale_factor=", scale_factor, "via", q_states.shape[-1] )
    attn_bias = np.zeros((L, S), dtype=np.float32 ) #dtype=q_states.dtype) # TODO: why bfloat16 not working?
    #print("apply_attention: attn_bias=", attn_bias.dtype)
    #print("apply_attention: attn bias=", #attn_bias,
    #      attn_bias.shape)
    temp_mask = np.tril( np.ones((L, S), dtype='bool'), k=0 )
    #print("apply_attention: temp mask=", #temp_mask, 
    #      temp_mask.shape)
    #attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))
    attn_bias = np.where( temp_mask, attn_bias, np.NINF)
    attn_bias = attn_bias.astype(q_states.dtype)
    #print("apply_attention: attn_bias", #attn_bias, 
    #      attn_bias.shape)
    #print("apply_attention: attn_bias", #attn_bias, 
    #      attn_bias.shape,attn_bias.dtype)
    #attn_weight = query @ key.transpose(-2, -1) * scale_factor
    #print(q_states.shape, k_states.shape, k_states.transpose((0,1,3,2)).shape)
    k_states_transposed = k_states.transpose((0,1,3,2))
    attn_weight_unscaled = np.matmul(q_states, k_states_transposed )
    attn_weight_unscaled = attn_weight_unscaled.astype( q_states.dtype)
    #print("attn_weight unscaled=", #attn_weight_unscaled, 
    #      attn_weight_unscaled.shape, attn_weight_unscaled.dtype)
    attn_weight =  attn_weight_unscaled * scale_factor
    #attn_weight += attn_bias
    attn_weight = np.add( attn_weight, attn_bias)
    attn_weight = attn_weight.astype( q_states.dtype ) # why need to coerce to bfloat16 ?
    #print("apply_attention: attn_weight=", #attn_weight, 
    #      attn_weight.shape)
    #attn_weight = torch.softmax(attn_weight, dim=-1)
    exp_ = np.exp(attn_weight)
    #print("apply_attention: exp=", #exp_,
    #      exp_.shape, exp_.dtype)
    sum_ = np.sum(np.exp(attn_weight), axis=3)
    #print("sum_", #sum_,
    #      sum_.shape)
    sum__ = np.expand_dims(sum_, 3)
    sum___ = np.broadcast_to( sum__, (1, 32, q_len, q_len) )
    #print("sum___", #sum___,
    #      sum___.shape)
    #raise Exception("stop!")
    sm = exp_ / sum___
    #print("apply_attention: softmax=", #sm,
    #      sm.shape, sm.dtype)
    check = np.sum(sm, axis=3)
    #print("apply_attention: check=", #check,
    #      check.shape, check.dtype)
    #HF: attn_weight = torch.dropout(attn_weight, dropout_p, train=True)
    #HF: attn = attn_weight @ value
    z_states = np.matmul( sm, v_states)
    z_states = z_states.astype( q_states.dtype ) # why need to coerce to bfloat16?
    #print("apply_attention: attn 1=", attn,
    #      attn.shape)
    #attn_output = attn_output.transpose(1, 2).contiguous()
    attn = z_states.transpose((0,2,1,3))
    #print("apply_attention: attn 2=", #attn,
    #      attn.shape)  
    #HF: attn_output = attn_output.view(bsz, q_len, -1)
    #print(batch_size, q_len, attn.shape[-1])
    attn = attn.reshape( (batch_size, q_len, attn.shape[-2]*attn.shape[-1] ) )
    #print("apply_attention: attn 3=", #attn,
    #      attn.shape)
    o_weights = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.attention.wo.weight' % layer] ]
    #print("apply_attention: o_weights=", #o_weights, 
    #      o_weights.shape, o_weights.dtype)  
    o_proj = np.matmul( attn, o_weights.transpose() )
    o_proj = o_proj.astype( q_states.dtype)
    #print("apply_attention: o_proj=", #o_proj, 
    #      o_proj.shape, o_proj.dtype)
    return o_proj, z_states

In [57]:

def apply_mlp( attn, hf_model, residual, layer ):
    # prepare for fully connected
    hidden_state = np.add( residual, attn )
    #print("apply_mlp: hidden_state after residual add=", hidden_state,
    #      hidden_state.shape, hidden_state.dtype)
    residual = hidden_state
    
    # Apply RMS Norm 
    hidden_state = hidden_state.astype(np.float32)
    #print("hidden_state_32=", #hidden_state, 
    #      hidden_state.dtype)
    variance_epsilon= 1e-05
    rmsnorm = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.ffn_norm.weight' % layer] ]
    #print("rmsnorm=", rmsnorm, 
    #      rmsnorm.shape)
    variance = np.mean( np.power(hidden_state,2), 2, keepdims=True )
    #print("variance=", variance, 
    #      variance.shape) 
    v_sqrt = np.sqrt( variance + variance_epsilon )
    #print("v_sqrt=", #v_sqrt, 
    #      v_sqrt.shape) 
    v_rsqrt = np.reciprocal( v_sqrt  )
    #print("v_rsqrt=", #v_rsqrt, 
    #      v_rsqrt.shape) 
    hidden_state = np.multiply( hidden_state, v_rsqrt )
    #print("hidden_state_mult=", #hidden_state, 
    #      hidden_state.shape)
    normalized_hidden_state = rmsnorm * hidden_state.astype( ml_dtypes.bfloat16 )
    #print("normalized_hidden_state=", #normalized_hidden_state, 
    #      normalized_hidden_state.shape, normalized_hidden_state.dtype)
    down = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.feed_forward.w2.weight' % layer] ]
    gate = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.feed_forward.w1.weight' % layer] ]
    up = hf_model[ TORCH_HF_KEY_MAPPING['layers.%d.feed_forward.w3.weight' % layer] ]
    #print("apply_mlp: mlp mats down=", down, down.shape,
    #      "gate=", gate, gate.shape, "up=", up, up.shape)
    after_gate = np.matmul(normalized_hidden_state, gate.transpose())
    after_gate = after_gate.astype( attn.dtype ) # why need coercion?
    #print("apply_mlp: after gate=",  after_gate,
    #      after_gate.shape, after_gate.dtype)
    after_sigmoid = 1.0/(1.0 + np.exp(-after_gate))
    #print("apply_mlp: after_sigmoid=",  after_sigmoid,
    #      after_sigmoid.shape, after_sigmoid.dtype)
    after_silu = np.multiply( after_gate, after_sigmoid)
    #print("apply_mlp: after_silu=",  after_silu,
    #      after_silu.shape, after_silu.dtype)
    after_up = np.matmul(normalized_hidden_state, up.transpose())
    after_up = after_up.astype( attn.dtype ) # why need coercion?
    #print("apply_mlp: after_up=",  after_up,
    #      after_up.shape, after_up.dtype)
    pre_down = np.multiply(after_silu, after_up)
    pre_down = pre_down.astype( attn.dtype ) # why need coercion?
    #print("apply_mlp: pre_down=",  pre_down,
    #      pre_down.shape, pre_down.dtype)
    after_down = np.matmul(pre_down, down.transpose())
    after_down = after_down.astype( attn.dtype ) # why need coercion?
    #print("apply_mlp: after_down=",  after_down,
    #      after_down.shape, after_down.dtype)
    mlp_out = np.add( residual, after_down )
    return mlp_out

In [58]:
def final_norm( hidden_state, hf_model ):
    hidden_state = hidden_state.astype(np.float32)
    #print("hidden_state_32=", #hidden_state, 
    #      hidden_state.dtype) 
    variance_epsilon= 1e-05
    rmsnorm = hf_model[ TORCH_HF_KEY_MAPPING['norm.weight'] ]
    #print("rmsnorm=", #rmsnorm, 
    #      rmsnorm.shape)
    variance = np.mean( np.power(hidden_state,2), 2, keepdims=True )
    #print("variance=", #variance, 
    #      variance.shape)
    v_sqrt = np.sqrt( variance + variance_epsilon )
    #print("v_sqrt=", #v_sqrt, 
    #      v_sqrt.shape)
    v_rsqrt = np.reciprocal( v_sqrt  )
    #print("v_rsqrt=", #v_rsqrt, 
    #      v_rsqrt.shape)
    hidden_state = np.multiply( hidden_state, v_rsqrt )
    #print("hidden_state_mult=", #hidden_state, 
    #      hidden_state.shape)
    normalized_hidden_state = rmsnorm * hidden_state.astype( ml_dtypes.bfloat16 )
    #print("apply_rms_norm_pre_attention: normalized_hidden_state=", #normalized_hidden_state, 
    #      normalized_hidden_state.shape,normalized_hidden_state.dtype)
    return normalized_hidden_state

In [59]:
def final_prediction(final_hidden_state, hf_model):
    lm_head = hf_model[ TORCH_HF_KEY_MAPPING['output.weight'] ]
    logits = np.matmul(final_hidden_state, lm_head.transpose())
    #print("GW final_logits logits=", logits,
    #    logits.shape, logits.dtype)
    last_logit = logits[-1,-1,:]
    #print("GW final_logits last_logit=", last_logit,
    #    last_logit.shape, last_logit.dtype) 
    # probs = nn.functional.softmax(next_token_scores, dim=-1)
    exp_ = np.exp(last_logit)
    #print("GW final_logits exp=", exp_, exp_.shape, exp_.dtype)
    sum_ = np.sum(np.exp(last_logit), axis=0)
    #print("GW final_logits sum=", sum_, sum_.shape, sum_.dtype)
    sm = exp_ / sum_
    #print("GW final_logits sum=", sm, sm.shape, sm.dtype)
    max_id = np.argmax(sm)
    #print("GW final_logits argmax=", max_id, sm[max_id])
    return max_id

In [61]:
# MAIN

create_model_key_mappings()
tokenizer = create_tokenizer()
# tokenize prompt
prompt = ['''<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'''
             '''Cutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n'''
             '''You are a pirate chatbot who always responds in pirate speak!'''
             '''<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'''
             '''Who are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n''']
token_ids, batch_size, q_len = tokenize_prompt(prompt, tokenizer)
print("main: initial token_ids=", token_ids, type(token_ids))

main: initial token_ids= [[128000 128006   9125 128007    271  38766   1303  33025   2696     25
    6790    220   2366     18    198  15724   2696     25    220   1627
   10263    220   2366     19    271   2675    527    264  55066   6369
    6465    889   2744  31680    304  55066   6604      0 128009 128006
     882 128007    271  15546    527    499     30 128009 128006  78191
  128007    271]] <class 'numpy.ndarray'>


In [62]:
# load models
hf_model = load_hf_export_model()
torch_model = load_torch_model()

In [63]:
#
# Forward all layers with decode limit
#
counter=0

while True:

    print("Getting next token...", end='')
    
    # create token embeddings
    token_embeddings = token_ids__2__token_embeddings(token_ids, hf_model)
    q_len = token_ids.shape[1]
    
    # initialize first layer input from token_embeddings
    hidden_state = token_embeddings
    
    for layer in range(NUM_LAYERS):
        
        print("%d-" %layer , end='')
        
        # apply pre-attention layer norm
        residual, normalized_hidden_state = apply_rms_norm_pre_attention( hidden_state, hf_model, layer)
        
        # apply qkv projections
        q_state, k_state, v_state = project_to_qkv( normalized_hidden_state, hf_model, batch_size, q_len, layer)
        
        # apply attention
        attn_out, z_state = apply_attention( q_state, k_state, v_state, hf_model, batch_size, q_len, layer)
        
        # apply MLP
        mlp_out = apply_mlp( attn_out, hf_model, residual, layer)
    
        # prepare for next layer in the loop
        hidden_state = mlp_out
    
    normalized_hidden_state = final_norm( hidden_state, hf_model )
    
    predicted_token_id = final_prediction( normalized_hidden_state, hf_model)
    #print("main: predicted=", predicted_token_id)

    token_ids = np.append(token_ids, np.array([[predicted_token_id]]) )
    print("new token_ids=", token_ids)
    token_ids = token_ids.reshape(1, token_ids.size)
    print("new token_ids shape=", token_ids.shape)
    #print("main: predicted=", predicted_token_id)

    new_text = tokenizer.decode(token_ids[0,:])
    print()
    print("Done. Here is the new_text-->\n", new_text,"<--")
    print()

    counter += 1
    if counter>1:
        break
        
print("Done.")

Getting next token...0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18-19-20-21-22-23-24-25-26-27-28-29-30-31-new token_ids= [128000 128006   9125 128007    271  38766   1303  33025   2696     25
   6790    220   2366     18    198  15724   2696     25    220   1627
  10263    220   2366     19    271   2675    527    264  55066   6369
   6465    889   2744  31680    304  55066   6604      0 128009 128006
    882 128007    271  15546    527    499     30 128009 128006  78191
 128007    271   9014]
new token_ids shape= (1, 53)

Done. Here is the new_text-->
 system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a pirate chatbot who always responds in pirate speak!user

Who are you?assistant

Arr <--

Getting next token...0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18-19-20-21-22-23-24-25-26-27-28-29-30-31-new token_ids= [128000 128006   9125 128007    271  38766   1303  33025   2696     25
   6790    220   2366     18    198  15724   2696     25    220   1627
  102